# 🩺 Clinical AI: 7B 4-Bit NF4 QLoRA Fine-Tuning & Evaluation Pipeline
### Domain-Specific Fine-Tuning on 5,000 USMLE Clinical Reasoning Cases
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/er3dedrw44i/clinical-llm-finetuning/blob/main/qlora_colab.ipynb)

---
## 🎯 Objectives:
1. **Quantization**: Load `Qwen/Qwen2.5-7B-Instruct` in **BitsAndBytes 4-bit NormalFloat4 (NF4)** + **Double Quantization** to reduce VRAM from $16\text{ GB} \rightarrow 4.5\text{ GB}$ ($>70\%$ memory reduction).
2. **PEFT / LoRA**: Inject Low-Rank Adapters ($r=16, \alpha=32$) targeting all attention and MLP projection modules.
3. **SFT Training**: Train on real clinical reasoning cases with **Strict Completion-Only Loss Masking (`-100`)**.
4. **Benchmarking**: Record GPU memory allocation (VRAM peak) and training loss progression.

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q -U torch transformers peft trl bitsandbytes accelerate datasets

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Please change your Colab runtime to GPU: Runtime -> Change runtime type -> T4 GPU")

### Step 2: Download Dataset & Configure 4-Bit NF4 Quantization

In [ ]:
import os
import json
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./qlora_7b_adapter"

# 1. Download 5,000 Real USMLE Clinical Cases
print("📥 Loading USMLE Medical Dataset from medalpaca/medical_meadow_medqa...")
raw_ds = load_dataset("medalpaca/medical_meadow_medqa", split="train")

curated_records = []
for item in raw_ds:
    inp = item.get("input", "").strip()
    out = item.get("output", "").strip()
    if len(inp) > 30 and len(out) > 0:
        curated_records.append({
            "instruction": "You are a clinical AI physician. Analyze the patient presentation, identify the diagnosis, and recommend the best evidence-based treatment option.",
            "input": inp,
            "output": out
        })
    if len(curated_records) >= 5000:
        break

# 80/20 Train/Test Split
train_size = int(len(curated_records) * 0.80)
train_data = curated_records[:train_size]
test_data = curated_records[train_size:]
print(f"✅ Dataset Ready: {len(train_data)} Train Samples, {len(test_data)} Test Samples.")

### Step 3: Load 7B Foundation Model in 4-Bit NF4

In [ ]:
# Configure 4-bit NormalFloat4 + Double Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f"📥 Loading {MODEL_NAME} in 4-bit NF4...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for k-bit training (freezes quantized base layers)
base_model = prepare_model_for_kbit_training(base_model)

vram_loaded = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"✅ 7B Model Loaded! VRAM Allocated: {vram_loaded:.2f} GB (Base Model 16GB -> 4.5GB!)")

### Step 4: Inject LoRA Adapters & Verify Parameter Reduction

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

qlora_model = get_peft_model(base_model, peft_config)
print("--- 📊 Parameter Statistics ---")
qlora_model.print_trainable_parameters()

### Step 5: Format Data with Completion-Only Loss Masking (-100)

In [ ]:
def format_completion_only_tokens(samples, tokenizer, max_length=256):
    input_ids_list, attention_mask_list, labels_list = [], [], []
    for item in samples:
        instruction = item["instruction"]
        input_text = item.get("input", "")
        output = item["output"]
        context_str = f"\nContext: {input_text}" if input_text else ""

        prompt_str = (
            "<|im_start|>system\n"
            "You are an expert Clinical Medicine AI assistant. Provide accurate, evidence-based guidance.<|im_end|>\n"
            f"<|im_start|>user\n{instruction}{context_str}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        full_str = prompt_str + f"{output}<|im_end|>"

        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        full_ids = tokenizer.encode(full_str, add_special_tokens=False)

        if len(full_ids) > max_length:
            full_ids = full_ids[:max_length]

        prompt_len = min(len(prompt_ids), len(full_ids))
        labels = [-100] * prompt_len + full_ids[prompt_len:]

        input_ids_list.append(full_ids)
        attention_mask_list.append([1] * len(full_ids))
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    })

train_dataset = format_completion_only_tokens(train_data, tokenizer, max_length=256)
eval_dataset = format_completion_only_tokens(test_data, tokenizer, max_length=256)
print(f"✅ Formatted with Completion Masking: {len(train_dataset)} Train, {len(eval_dataset)} Test.")

### Step 6: Execute 4-Bit QLoRA SFT Training Loop

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=qlora_model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir="./qlora_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=qlora_model,
    train_dataset=train_dataset,
    data_collator=data_collator,
    args=training_args
)

print("🚀 Starting 4-bit QLoRA Training on NVIDIA GPU...")
train_result = trainer.train()

# Save trained adapter
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

peak_vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print("=" * 60)
print(f"🎉 QLoRA Training Complete! Final Loss: {train_result.metrics.get('train_loss', 0.0):.4f}")
print(f"📊 Peak VRAM Consumed: {peak_vram:.2f} GB (Fits on a single free T4 GPU!)")
print(f"💾 Trained 7B Adapter Saved to: {OUTPUT_DIR}")
print("=" * 60)

### Step 7: Live Clinical Inference with Merged 7B Model

In [ ]:
test_prompt = "A 62-year-old male with chronic kidney disease (eGFR 26 mL/min) and type 2 diabetes presents for glycemic management. Which first-line agent is contraindicated, and what alternative should be recommended?"

formatted_input = (
    "<|im_start|>system\n"
    "You are an expert Clinical Medicine AI assistant. Provide accurate, evidence-based guidance.<|im_end|>\n"
    f"<|im_start|>user\n{test_prompt}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
    output = qlora_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("🩺 [Fine-Tuned 7B QLoRA Response]:\n", response.strip())